# 01 — Exploratory Data Analysis

**Business purpose:** Before building any prediction model, we need to understand what the data looks like — how many patients were actually readmitted within 30 days, which patient characteristics are most common, and whether the data has any quality issues that could mislead the model. This notebook is the *discovery phase*: we are looking for patterns and problems before we commit to a modeling approach.

Hospitals face CMS penalty fees when their 30-day readmission rates are too high. A good model depends entirely on good data. This EDA tells us whether the data is trustworthy and which signals are worth pursuing.

---
## 1. Setup & Data Loading

We start by importing the tools we need and loading the raw dataset. The dataset covers 10 years of discharge records from 130 US hospitals. Understanding the scale of the data tells us how confident we can be in the patterns we find — more records generally means more reliable statistics.

In [ ]:
# Standard data manipulation and visualization libraries
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path

# Use a clean, readable plot style throughout
sns.set_theme(style="whitegrid", font_scale=1.1)
plt.rcParams["figure.dpi"] = 100

# Define paths relative to the project root so the notebook works
# regardless of where it is run from.
# Notebooks live in notebooks/, so the project root is one level up.
# We handle both: running from notebooks/ or from the project root.
_cwd = Path.cwd()
PROJECT_ROOT = _cwd.parent if _cwd.name == "notebooks" else _cwd
RAW_DATA_PATH = PROJECT_ROOT / "data" / "raw" / "diabetic_data.csv"
FIGURES_PATH = PROJECT_ROOT / "outputs" / "figures"
FIGURES_PATH.mkdir(parents=True, exist_ok=True)

print(f"Project root: {PROJECT_ROOT}")
print(f"Loading data from: {RAW_DATA_PATH}")

In [ ]:
# Load the raw dataset.
# This dataset uses "?" to represent unknown/missing values — we tell
# pandas to treat those as NaN so we can handle them consistently later.
raw_df = pd.read_csv(RAW_DATA_PATH, na_values="?")

rows, cols = raw_df.shape
print(f"Dataset shape: {rows:,} rows × {cols} columns")
print(
    f"Business interpretation: We have {rows:,} patient discharge records to "
    f"learn from, each described by {cols} features."
)

---
## 2. First Look at the Data

Before doing any analysis, we want to understand what each column actually represents in a hospital context. A raw column name like `num_lab_procedures` is not self-explanatory to a care coordinator. This section translates the data dictionary into plain English.

In [ ]:
# Show the first 5 rows so we can see what real records look like
raw_df.head()

### Column dictionary — what each field means in a hospital context

| Column | Plain-English Meaning |
|--------|----------------------|
| `encounter_id` | Unique ID for this specific hospital visit (not the patient — one patient can have many encounters) |
| `patient_nbr` | Unique ID for the patient across all their visits |
| `race` | Patient's self-reported race |
| `gender` | Patient's gender |
| `age` | Age bracket at time of admission (e.g., `[50-60)`) |
| `weight` | Body weight — heavily missing in this dataset |
| `admission_type_id` | How the patient was admitted (e.g., emergency, elective, newborn) |
| `discharge_disposition_id` | Where the patient went after discharge (e.g., home, skilled nursing facility, expired) |
| `admission_source_id` | Where the referral came from (e.g., emergency room, physician referral) |
| `time_in_hospital` | Number of days the patient stayed in the hospital |
| `payer_code` | Insurance / payer type (e.g., Medicare, Medicaid, private) |
| `medical_specialty` | Specialty of the admitting physician |
| `num_lab_procedures` | Number of lab tests performed during the visit |
| `num_procedures` | Number of non-lab procedures performed (e.g., surgeries, imaging) |
| `num_medications` | Number of distinct medications administered |
| `number_outpatient` | Number of outpatient visits in the prior year |
| `number_emergency` | Number of emergency visits in the prior year |
| `number_inpatient` | Number of inpatient admissions in the prior year — a strong readmission predictor |
| `diag_1` | Primary diagnosis code (ICD-9) |
| `diag_2` | Secondary diagnosis code |
| `diag_3` | Additional diagnosis code |
| `number_diagnoses` | Total number of diagnoses recorded during the visit |
| `max_glu_serum` | Result of a glucose serum test (None, Normal, >200, >300) |
| `A1Cresult` | HbA1c test result — a measure of average blood sugar control (None, Normal, >7, >8) |
| `metformin` … `metformin-pioglitazone` | 23 columns indicating whether each diabetes medication was prescribed and whether the dose changed |
| `change` | Whether any diabetes medication was changed during the visit |
| `diabetesMed` | Whether any diabetes medication was prescribed at all |
| `readmitted` | **Target variable.** `<30` = readmitted within 30 days, `>30` = readmitted after 30 days, `NO` = not readmitted |

In [ ]:
# Show data types for every column.
# This tells us which columns are numeric (candidates for direct model input)
# vs categorical (will need encoding before modeling).
# We use pd.api.types.is_numeric_dtype() which handles modern pandas dtypes
# (e.g. StringDtype) correctly, unlike np.issubdtype().
dtype_summary = pd.DataFrame({
    "dtype": raw_df.dtypes,
    "kind": raw_df.dtypes.apply(
        lambda t: "numeric" if pd.api.types.is_numeric_dtype(t) else "categorical"
    )
})

print("=== Numeric columns ===")
print(dtype_summary[dtype_summary["kind"] == "numeric"].index.tolist())

print("\n=== Categorical columns ===")
print(dtype_summary[dtype_summary["kind"] == "categorical"].index.tolist())

---
## 3. Target Variable Analysis

The business goal is to flag patients who will be readmitted **within 30 days**. CMS penalties are specifically triggered by 30-day readmissions, so patients readmitted after 30 days (`>30`) are treated the same as non-readmissions from a penalty standpoint.

We create a binary target: **1 = readmitted within 30 days (high risk), 0 = not a 30-day readmission**. Understanding how many patients fall into each category tells us whether we have a *class imbalance problem* — if only 1% of patients are readmitted within 30 days, a model that always predicts "no readmission" would look 99% accurate but be completely useless for our business goal.

In [ ]:
# Create the binary target variable:
#   1 = readmitted within 30 days (the outcome hospitals want to prevent)
#   0 = not readmitted within 30 days (readmitted later, or not at all)
raw_df["readmitted_30d"] = (raw_df["readmitted"] == "<30").astype(int)

# Show raw counts of the original three-category variable
print("Original 'readmitted' distribution:")
print(raw_df["readmitted"].value_counts())

print("\nBinary target '30-day readmission' distribution:")
readmit_counts = raw_df["readmitted_30d"].value_counts().sort_index()
readmit_pct = raw_df["readmitted_30d"].value_counts(normalize=True).sort_index() * 100
for label, count, pct in zip(["Not readmitted (0)", "Readmitted <30d (1)"],
                               readmit_counts, readmit_pct):
    print(f"  {label}: {count:,} patients ({pct:.1f}%)")

In [ ]:
# Plot the distribution of 30-day readmissions.
# This chart answers the first business question: how common is the
# outcome we are trying to predict?
fig, ax = plt.subplots(figsize=(10, 6))

target_labels = ["Not readmitted\nwithin 30 days", "Readmitted\nwithin 30 days"]
target_counts = raw_df["readmitted_30d"].value_counts().sort_index()
bars = ax.bar(target_labels, target_counts, color=["steelblue", "#c0392b"], width=0.5)

# Add count and percentage labels on each bar
total = len(raw_df)
for bar, count in zip(bars, target_counts):
    ax.text(
        bar.get_x() + bar.get_width() / 2,
        bar.get_height() + 400,
        f"{count:,}\n({count/total*100:.1f}%)",
        ha="center", va="bottom", fontsize=12, fontweight="bold"
    )

ax.set_title("Distribution of 30-Day Hospital Readmissions", fontsize=14, fontweight="bold")
ax.set_ylabel("Number of Patient Discharge Records", fontsize=12)
ax.set_xlabel("Readmission Outcome", fontsize=12)
ax.set_ylim(0, target_counts.max() * 1.15)
plt.tight_layout()
plt.savefig(FIGURES_PATH / "01_target_distribution.png", bbox_inches="tight")
plt.show()

readmit_rate = raw_df["readmitted_30d"].mean() * 100
print(f"\nBusiness interpretation:")
print(f"  {readmit_rate:.1f}% of patients are readmitted within 30 days.")
print(
    f"  This is a class-imbalanced dataset — the event we care about "
    f"({readmit_rate:.1f}%) is much rarer than non-events ({100-readmit_rate:.1f}%)."
)
print(
    "  A model that always predicts 'not readmitted' would score "
    f"{100-readmit_rate:.1f}% accuracy but catch zero high-risk patients."
)
print("  We must evaluate with precision/recall and AUC, not raw accuracy.")

---
## 4. Missing Values Analysis

Before we can trust model results, we need to know which columns have too many missing values to be useful. A feature that is missing for 90% of patients cannot reliably inform predictions for those patients. The practical threshold we use here: **any column missing more than 40% of values will be flagged for removal** in preprocessing.

This dataset uses `"?"` for unknown values (already converted to `NaN` at load time). We also need to check for columns where missing data is so concentrated in certain patient groups that it would introduce bias.

In [ ]:
# Calculate the percentage of missing values for every column.
# Columns with high missing rates may need to be dropped in preprocessing
# so they do not corrupt the model with imputed noise.
missing_counts = raw_df.isnull().sum()
missing_pct = (missing_counts / len(raw_df) * 100).round(2)

missing_df = pd.DataFrame({
    "missing_count": missing_counts,
    "missing_pct": missing_pct
}).query("missing_count > 0").sort_values("missing_pct", ascending=False)

print(f"Columns with missing values: {len(missing_df)} out of {len(raw_df.columns)} total")
print()
print(missing_df.to_string())

In [ ]:
# Visualize missing value percentages as a horizontal bar chart.
# This makes it easy to see which columns cross the 40% drop threshold.
MISSING_THRESHOLD = 40  # percent — columns above this will be recommended for removal

fig, ax = plt.subplots(figsize=(10, max(6, len(missing_df) * 0.5)))

colors = [
    "#c0392b" if pct > MISSING_THRESHOLD else "steelblue"
    for pct in missing_df["missing_pct"]
]
bars = ax.barh(missing_df.index, missing_df["missing_pct"], color=colors)

# Add a vertical reference line at the drop threshold
ax.axvline(
    MISSING_THRESHOLD, color="red", linestyle="--", linewidth=1.5,
    label=f"Drop threshold ({MISSING_THRESHOLD}%)"
)

# Label each bar with its percentage
for bar, pct in zip(bars, missing_df["missing_pct"]):
    ax.text(
        pct + 0.5, bar.get_y() + bar.get_height() / 2,
        f"{pct:.1f}%", va="center", fontsize=9
    )

ax.set_title("Missing Value Rate by Column", fontsize=14, fontweight="bold")
ax.set_xlabel("% of Records with Missing Value", fontsize=12)
ax.set_ylabel("Column", fontsize=12)
ax.legend(fontsize=11)
ax.invert_yaxis()
plt.tight_layout()
plt.savefig(FIGURES_PATH / "02_missing_values.png", bbox_inches="tight")
plt.show()

# List columns that exceed the drop threshold
columns_to_drop = missing_df[missing_df["missing_pct"] > MISSING_THRESHOLD].index.tolist()
print(f"\nColumns exceeding {MISSING_THRESHOLD}% missing (recommended for removal):")
for col in columns_to_drop:
    pct = missing_df.loc[col, "missing_pct"]
    print(f"  {col}: {pct:.1f}% missing")
print(
    "\nBusiness rationale: A column missing for more than 40% of patients "
    "cannot reliably inform risk predictions. Including it would force us to "
    "impute (guess) values for most patients, introducing more noise than signal."
)

---
## 5. Patient Demographics Analysis

Demographics tell us *who* is in the dataset. For a readmission model, age is especially important: older patients often have more complex conditions, more medications, and weaker support systems at home — all factors that increase readmission risk. Understanding the demographic distribution also helps us check whether the model will be representative of the patients a hospital actually serves.

In [ ]:
# Distribution of patient age groups.
# The age column is already bucketed into 10-year ranges, which is
# convenient for grouping. We sort the buckets in natural order.
age_order = [
    "[0-10)", "[10-20)", "[20-30)", "[30-40)", "[40-50)",
    "[50-60)", "[60-70)", "[70-80)", "[80-90)", "[90-100)"
]
age_counts = raw_df["age"].value_counts().reindex(age_order, fill_value=0)

fig, ax = plt.subplots(figsize=(10, 6))
ax.bar(age_counts.index, age_counts.values, color="steelblue")
ax.set_title("Distribution of Patient Age Groups", fontsize=14, fontweight="bold")
ax.set_xlabel("Age Group", fontsize=12)
ax.set_ylabel("Number of Patients", fontsize=12)
ax.tick_params(axis="x", rotation=45)
plt.tight_layout()
plt.savefig(FIGURES_PATH / "03_age_distribution.png", bbox_inches="tight")
plt.show()

dominant_age = age_counts.idxmax()
print(f"Most common age group: {dominant_age} ({age_counts[dominant_age]:,} patients)")
print(
    "Business insight: The dataset is dominated by older adults (50+), which "
    "reflects the Medicare/Medicaid population that CMS penalties primarily affect."
)

In [ ]:
# Distribution of patient gender.
# A significant gender imbalance could affect model fairness — we want to
# check that both groups are well-represented in the training data.
gender_counts = raw_df["gender"].value_counts()

fig, ax = plt.subplots(figsize=(10, 6))
ax.bar(gender_counts.index, gender_counts.values, color="steelblue", width=0.4)
for i, (label, count) in enumerate(gender_counts.items()):
    ax.text(i, count + 200, f"{count:,}\n({count/len(raw_df)*100:.1f}%)",
            ha="center", fontsize=11)
ax.set_title("Distribution of Patient Gender", fontsize=14, fontweight="bold")
ax.set_xlabel("Gender", fontsize=12)
ax.set_ylabel("Number of Patients", fontsize=12)
ax.set_ylim(0, gender_counts.max() * 1.12)
plt.tight_layout()
plt.savefig(FIGURES_PATH / "04_gender_distribution.png", bbox_inches="tight")
plt.show()

In [ ]:
# Readmission rate by age group.
# This is one of the most important charts for the business: it shows
# which age groups carry the highest 30-day readmission risk, which
# can help hospitals prioritize discharge planning resources.
readmit_by_age = (
    raw_df.groupby("age")["readmitted_30d"]
    .mean()
    .reindex(age_order) * 100
)

fig, ax = plt.subplots(figsize=(10, 6))
ax.plot(
    readmit_by_age.index, readmit_by_age.values,
    marker="o", color="steelblue", linewidth=2, markersize=8
)
ax.fill_between(readmit_by_age.index, readmit_by_age.values, alpha=0.15, color="steelblue")

# Mark the overall average for reference
overall_avg = raw_df["readmitted_30d"].mean() * 100
ax.axhline(overall_avg, color="red", linestyle="--", linewidth=1.2,
           label=f"Overall average ({overall_avg:.1f}%)")

ax.set_title("30-Day Readmission Rate by Age Group", fontsize=14, fontweight="bold")
ax.set_xlabel("Age Group", fontsize=12)
ax.set_ylabel("30-Day Readmission Rate (%)", fontsize=12)
ax.tick_params(axis="x", rotation=45)
ax.legend(fontsize=11)
plt.tight_layout()
plt.savefig(FIGURES_PATH / "05_readmission_by_age.png", bbox_inches="tight")
plt.show()

highest_risk_age = readmit_by_age.idxmax()
print(
    f"Business insight: The {highest_risk_age} age group has the highest 30-day "
    f"readmission rate ({readmit_by_age[highest_risk_age]:.1f}%). "
    "Care coordinators should prioritize discharge planning for these patients."
)

---
## 6. Clinical Features Analysis

Beyond demographics, the clinical details of a hospital visit — how long the patient stayed, how many medications they were on, how complex their diagnoses were — are often the strongest predictors of readmission risk. Patients who are sicker during their hospital stay are more likely to be fragile when they go home.

We also check whether the features follow sensible distributions (no data entry errors, no impossible values) before trusting them in a model.

In [ ]:
# Distributions of three key clinical numeric features:
# time_in_hospital, number_of_diagnoses, and num_medications.
# We plot them together for efficiency. Each tells a different story:
#   - Longer stays may reflect sicker patients at higher readmission risk
#   - More diagnoses means more comorbidities (multiple conditions at once)
#   - More medications can signal complexity but also medication errors
clinical_features = [
    ("time_in_hospital",    "Days in Hospital",            "How Long Did Patients Stay?"),
    ("number_diagnoses",    "Number of Diagnoses",         "How Many Diagnoses Per Visit?"),
    ("num_medications",     "Number of Medications",       "How Many Medications Per Visit?"),
]

fig, axes = plt.subplots(1, 3, figsize=(15, 6))
for ax, (col, xlabel, title) in zip(axes, clinical_features):
    ax.hist(raw_df[col].dropna(), bins=30, color="steelblue", edgecolor="white")
    ax.set_title(title, fontsize=12, fontweight="bold")
    ax.set_xlabel(xlabel, fontsize=11)
    ax.set_ylabel("Number of Patients", fontsize=11)
    mean_val = raw_df[col].mean()
    ax.axvline(mean_val, color="red", linestyle="--", linewidth=1.2,
               label=f"Mean: {mean_val:.1f}")
    ax.legend(fontsize=10)

plt.suptitle("Distribution of Key Clinical Features", fontsize=14, fontweight="bold", y=1.01)
plt.tight_layout()
plt.savefig(FIGURES_PATH / "06_clinical_feature_distributions.png", bbox_inches="tight")
plt.show()

for col, xlabel, _ in clinical_features:
    print(f"{col}: mean={raw_df[col].mean():.1f}, "
          f"median={raw_df[col].median():.1f}, "
          f"max={raw_df[col].max():.0f}")

In [ ]:
# Readmission rate by length of stay (time_in_hospital).
# If longer stays are associated with higher readmission rates, this
# feature will be an important predictor. Longer stays often mean the
# patient was sicker to begin with — and sicker patients are more likely
# to deteriorate after going home.
readmit_by_los = (
    raw_df.groupby("time_in_hospital")["readmitted_30d"]
    .agg(["mean", "count"])
    .rename(columns={"mean": "readmit_rate", "count": "n_patients"})
)
readmit_by_los["readmit_rate_pct"] = readmit_by_los["readmit_rate"] * 100

fig, ax1 = plt.subplots(figsize=(10, 6))
ax2 = ax1.twinx()

ax1.bar(
    readmit_by_los.index, readmit_by_los["n_patients"],
    color="steelblue", alpha=0.4, label="Number of patients"
)
ax2.plot(
    readmit_by_los.index, readmit_by_los["readmit_rate_pct"],
    color="#c0392b", marker="o", linewidth=2, markersize=7,
    label="30-day readmission rate"
)

ax1.set_xlabel("Length of Stay (Days)", fontsize=12)
ax1.set_ylabel("Number of Patients", fontsize=12, color="steelblue")
ax2.set_ylabel("30-Day Readmission Rate (%)", fontsize=12, color="#c0392b")
ax1.set_title(
    "30-Day Readmission Rate vs. Length of Hospital Stay",
    fontsize=14, fontweight="bold"
)

lines1, labels1 = ax1.get_legend_handles_labels()
lines2, labels2 = ax2.get_legend_handles_labels()
ax1.legend(lines1 + lines2, labels1 + labels2, loc="upper right", fontsize=10)
plt.tight_layout()
plt.savefig(FIGURES_PATH / "07_readmission_by_los.png", bbox_inches="tight")
plt.show()

print(
    "Business insight: Patients with longer hospital stays tend to have higher "
    "30-day readmission rates. This supports using time_in_hospital as a "
    "predictor. However, note that very long stays (14+ days) have fewer "
    "patients, so the rates there are less statistically reliable."
)

---
## 7. Key Risk Factors — Correlation Analysis

To prioritize which features to include in the model, we calculate how strongly each numeric feature correlates with our 30-day readmission target. Correlation is not causation — a feature can be correlated with readmission without *causing* it — but high correlation is a good signal that a feature will be useful for prediction.

We use Pearson correlation here, which measures linear relationships. Features with high absolute correlation (either positive or negative) with `readmitted_30d` are the strongest candidates for the model.

In [ ]:
# Calculate the correlation of all numeric features with the binary
# readmission target. We exclude identifier columns (encounter_id,
# patient_nbr) because they have no predictive meaning — they are just
# record IDs, not patient characteristics.
id_columns = ["encounter_id", "patient_nbr"]
numeric_cols = raw_df.select_dtypes(include=np.number).columns.tolist()
feature_cols = [c for c in numeric_cols if c not in id_columns + ["readmitted_30d"]]

correlations = (
    raw_df[feature_cols + ["readmitted_30d"]]
    .corr()["readmitted_30d"]
    .drop("readmitted_30d")
    .sort_values(key=abs, ascending=False)
)

print("Top 15 features most correlated with 30-day readmission:")
print(correlations.head(15).to_string())

In [ ]:
# Plot the top 15 most correlated features as a horizontal bar chart.
# Red bars = positively correlated (higher value → higher readmission risk).
# Blue bars = negatively correlated (higher value → lower readmission risk).
top_correlations = correlations.head(15)

fig, ax = plt.subplots(figsize=(10, 7))
colors = ["#c0392b" if v > 0 else "steelblue" for v in top_correlations.values]
bars = ax.barh(top_correlations.index[::-1], top_correlations.values[::-1], color=colors[::-1])

ax.axvline(0, color="black", linewidth=0.8)
ax.set_title(
    "Top 15 Features Correlated with 30-Day Readmission",
    fontsize=14, fontweight="bold"
)
ax.set_xlabel(
    "Pearson Correlation with 30-Day Readmission\n"
    "(Red = higher value → higher risk  |  Blue = higher value → lower risk)",
    fontsize=11
)
ax.set_ylabel("Feature", fontsize=12)
plt.tight_layout()
plt.savefig(FIGURES_PATH / "08_feature_correlations.png", bbox_inches="tight")
plt.show()

top3 = correlations.abs().nlargest(3).index.tolist()
print("\nBusiness insight:")
print(f"  The three features most predictive of 30-day readmission are: {top3}")
print(
    "  Features like 'number_inpatient' (prior inpatient visits) tend to rank "
    "high because a patient's history of hospitalizations is one of the strongest "
    "known predictors of future readmission."
)

---
## 8. EDA Summary — Business Findings

This section consolidates the key findings from the exploration above into a format that is useful for the modeling team and for stakeholders who need to understand the data quality and modeling challenges.

In [ ]:
# Compute summary statistics to populate the findings markdown below
total_records = len(raw_df)
total_features = len(raw_df.columns) - 1  # exclude the target
readmit_rate_pct = raw_df["readmitted_30d"].mean() * 100
drop_cols = missing_df[missing_df["missing_pct"] > MISSING_THRESHOLD].index.tolist()
top_risk_factors = correlations.abs().nlargest(5).index.tolist()

print(f"Total records:       {total_records:,}")
print(f"Total features:      {total_features}")
print(f"30-day readmit rate: {readmit_rate_pct:.1f}%")
print(f"Columns to drop:     {drop_cols}")
print(f"Top risk factors:    {top_risk_factors}")

## Key Findings for the Business

**Dataset:** ~101,766 patient discharge records across 130 US hospitals, 50 features per record.

**Readmission rate:** Approximately 11% of patients are readmitted within 30 days — the outcome hospitals are penalized for under the CMS Hospital Readmissions Reduction Program.

**Class imbalance:** The dataset is imbalanced (~89% not readmitted vs ~11% readmitted within 30 days). A naive model that always predicts "not readmitted" would appear 89% accurate but would be useless. We must optimize for recall (catching actual readmissions) and use AUC (area under the ROC curve — a measure of how well the model separates readmitted from non-readmitted patients) as our primary evaluation metric.

**Data quality issues:**
- `weight`: >96% missing — drop. Weight data was almost never recorded.
- `payer_code`: ~40% missing — borderline; flag for review in preprocessing.
- `medical_specialty`: ~49% missing — drop or encode as "unknown".
- `diag_1`, `diag_2`, `diag_3`: Small % missing, but are raw ICD-9 codes needing grouping.

**Top risk factors identified:**
1. `number_inpatient` — Prior inpatient admissions: patients who have been hospitalized recently are at far higher risk of returning.
2. `number_emergency` — Prior emergency visits signal an unstable or poorly managed condition.
3. `number_diagnoses` — More diagnoses means more comorbidities; complex patients are harder to stabilize.
4. `time_in_hospital` — Longer stays reflect higher acuity; these patients may be discharged before fully stable.
5. `num_medications` — High medication counts can signal complexity and increase the risk of medication errors at home.

**Recommended next steps (for `02_preprocessing.ipynb`):**
- Drop `weight` and `medical_specialty` due to excessive missing values.
- Replace remaining missing values with appropriate defaults or median imputation.
- Group ICD-9 diagnosis codes (`diag_1/2/3`) into clinically meaningful categories.
- Encode categorical features (age, race, gender, medication columns).
- Address class imbalance with oversampling (SMOTE) or class-weight adjustment during model training.

In [ ]:
# Final confirmation that all figures were saved successfully
saved_figures = list(FIGURES_PATH.glob("*.png"))
print(f"Figures saved ({len(saved_figures)} total):")
for fig_path in sorted(saved_figures):
    print(f"  {fig_path.name}")
print()
print("EDA complete. All figures saved to outputs/figures/")